# v11 by hand

start over from v5

In [1]:
# Cell 1: Install and Import Dependencies
!pip install datasets gensim scikit-learn pandas numpy matplotlib seaborn pillow tqdm

import pandas as pd
import numpy as np
from datasets import load_dataset
from collections import defaultdict, Counter
import json
import re
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

print("✓ Dependencies loaded")


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
✓ Dependencies loaded


# Cell 2: Load BBAW Dataset (100K translation pairs)

In [2]:

print("Loading BBAW dataset...")
bbaw_dataset = load_dataset("phiwi/bbaw_egyptian")

# Convert to pandas for easier manipulation
bbaw_df = pd.DataFrame(bbaw_dataset['train'])

print(f"BBAW Dataset loaded:")
print(f"  - Total examples: {len(bbaw_df):,}")
print(f"  - With hieroglyphs: {(bbaw_df['hieroglyphs'] != '').sum():,}")
print(f"  - Transcription only: {(bbaw_df['hieroglyphs'] == '').sum():,}")

# Show sample
print("\nSample entry:")
sample = bbaw_df.iloc[100]
print(f"Transcription: {sample['transcription']}")
print(f"Translation (DE): {sample['translation']}")
print(f"Hieroglyphs: {sample['hieroglyphs'][:50]}...")

Loading BBAW dataset...
BBAW Dataset loaded:
  - Total examples: 100,736
  - With hieroglyphs: 35,510
  - Transcription only: 65,226

Sample entry:
Transcription: sfḫ qn =k   
Translation (DE): Lass (wörtl.: wehre ab) deine Böswilligkeit (hinter dir)! ...
Hieroglyphs: ...


# Cell 2.5: Load Ramses Online Dataset (Scraped)

In [3]:


def load_ramses_data(json_path):
    """Load scraped Ramses Online data"""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return pd.DataFrame(data)

# Load Ramses data (update path as needed)
ramses_path = "../../heiro_v5_getdata/data/raw/ramses_raw.json"  # Adjust path
try:
    ramses_df = load_ramses_data(ramses_path)
    
    print(f"Ramses Online Dataset loaded:")
    print(f"  - Total examples: {len(ramses_df):,}")
    print(f"  - Period: {ramses_df['metadata'].apply(lambda x: x.get('period', 'Unknown')).value_counts().to_dict()}")
    
    # Show sample
    print("\nSample entry:")
    sample = ramses_df.iloc[10]
    print(f"Transliteration: {sample['transliteration']}")
    print(f"Translation (FR): {sample['translation']}")
    print(f"Period: {sample['metadata']['period']}")
    print(f"Dating: {sample['metadata']['dating']}")
    
except FileNotFoundError:
    print(f"⚠ Ramses data not found at {ramses_path}")
    ramses_df = None

Ramses Online Dataset loaded:
  - Total examples: 9,644
  - Period: {'Late Egyptian': 9644}

Sample entry:
Transliteration: jw m jrj šmj.t r wꜣḥ wꜥ wꜥ(.tw) m m w
Translation (FR): N'allez pas laisser là un seul d'entres eux.
Period: Late Egyptian
Dating: Ramsès XI


# Cell 3: Load TLA Premium Dataset (12K high-quality sentences)

In [4]:

print("Loading TLA Premium dataset...")
tla_dataset = load_dataset("thesaurus-linguae-aegyptiae/tla-Earlier_Egyptian_original-v18-premium")

# Convert to pandas
tla_df = pd.DataFrame(tla_dataset['train'])

print(f"\nTLA Premium Dataset loaded:")
print(f"  - Total examples: {len(tla_df):,}")
print(f"  - All have hieroglyphs: {(tla_df['hieroglyphs'] != '').sum():,}")
print(f"  - All fully lemmatized: ✓")

# Show sample with linguistic annotations
print("\nSample entry:")
sample = tla_df.iloc[100]
print(f"Hieroglyphs: {sample['hieroglyphs']}")
print(f"Transliteration: {sample['transliteration']}")
print(f"Lemmatization: {sample['lemmatization']}")
print(f"POS Tags: {sample['UPOS']}")
print(f"Glossing: {sample['glossing']}")
print(f"Translation (DE): {sample['translation']}")
print(f"Date range: {sample['dateNotBefore']} to {sample['dateNotAfter']}")

Loading TLA Premium dataset...

TLA Premium Dataset loaded:
  - Total examples: 12,773
  - All have hieroglyphs: 12,773
  - All fully lemmatized: ✓

Sample entry:
Hieroglyphs: 𓏙 𓈖 𓍹𓊪𓊪𓇋𓇋𓍺 𓍹𓇳𓄤𓂓𓍺 𓅨𓂋 𓇬𓈖<g>D82</g>𓏘𓏘 𓍹𓊪𓊪𓇋𓇋𓍺 𓍹𓇳𓄤𓂓𓍺 𓇋𓂋 𓏙𓏙 𓎡
Transliteration: ꞽmi̯ n ppy nfr-kꜣ-rꜥw wr wnm ppy nfr-kꜣ-rꜥw ꞽr ḏḏ =k
Lemmatization: 25180|ꞽmi̯ 400055|n 400313|Ppy 400330|Nfr-kꜣ-Rꜥw 47300|wr 46710|wnm 400313|Ppy 400330|Nfr-kꜣ-Rꜥw 91903|r 96700|rḏi̯ 10110|=k
POS Tags: VERB ADP PROPN PROPN ADV VERB PROPN PROPN ADP VERB PRON
Glossing: V\imp.sg PREP ROYLN ROYLN ADV V\tam.act ROYLN ROYLN PREP V~ipfv.act:stpr -2sg.m
Translation (DE): Gib dem Pepi Neferkare reichlich, damit Pepi Neferkare soviel du gibst essen kann.
Date range: -2278 to -2184


In [5]:
# Cell 4: Parse Lexicon.txt from HamdiJr
# You'll need to download this file from: 
# https://huggingface.co/datasets/HamdiJr/Egyptian_hieroglyphs

def parse_lexicon(lexicon_path):
    """
    Parse lexicon.txt format:
    Gardiner_codes,;transliteration;English_translation;occurrence_score;
    """
    lexicon = []
    
    with open(lexicon_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            parts = line.split(';')
            if len(parts) >= 4:
                gardiner_codes = parts[0].strip(',')
                transliteration = parts[1]
                meanings = parts[2]
                occurrence = float(parts[3]) if parts[3] else 0.0
                
                lexicon.append({
                    'gardiner_codes': gardiner_codes,
                    'gardiner_list': [g.strip() for g in gardiner_codes.split(',') if g.strip()],
                    'transliteration': transliteration,
                    'meanings_en': meanings,
                    'occurrence': occurrence
                })
    
    return pd.DataFrame(lexicon)

# Load lexicon (update path as needed)
lexicon_path = '../../heiro_v10_refinement/data/Lexicon.txt'
try:
    lexicon_df = parse_lexicon(lexicon_path)
    print(f"Lexicon loaded: {len(lexicon_df):,} entries")
    
    # Statistics
    print(f"\nLexicon Statistics:")
    print(f"  - Unique Gardiner signs: {len(set([g for gl in lexicon_df['gardiner_list'] for g in gl])):,}")
    print(f"  - High confidence (occurrence > 10): {(lexicon_df['occurrence'] > 10).sum():,}")
    print(f"  - Medium confidence (1-10): {((lexicon_df['occurrence'] > 1) & (lexicon_df['occurrence'] <= 10)).sum():,}")
    print(f"  - Low confidence (0-1): {(lexicon_df['occurrence'] <= 1).sum():,}")
    
    # Show top entries by occurrence
    print("\nMost frequent entries:")
    print(lexicon_df.nlargest(10, 'occurrence')[['transliteration', 'meanings_en', 'occurrence']])
    
except FileNotFoundError:
    print(f"⚠ Lexicon file not found at {lexicon_path}")
    print("Please download from HamdiJr/Egyptian_hieroglyphs dataset")
    lexicon_df = None

Lexicon loaded: 11,727 entries

Lexicon Statistics:
  - Unique Gardiner signs: 1,134
  - High confidence (occurrence > 10): 240
  - Medium confidence (1-10): 474
  - Low confidence (0-1): 11,013

Most frequent entries:
      transliteration                    meanings_en  occurrence
4504                i                        I,me,my    1543.850
5130               =i                        I,me,my    1543.850
11710               W                   same as G43*    1051.000
11711               W                   same as G43*    1051.000
25                swr                          drink     837.000
11219               t                      you, your     654.555
11437               T                       you,your     654.555
11465              =t                      you, your     654.555
2449                A  part\u00edcula encl\u00edtica     627.067
2450                A     vulture, bird (en general)     627.067


# Cell 5: Parse N-grams.txt from HamdiJr

In [6]:


def parse_ngrams(ngrams_path):
    """
    Parse ngrams.txt format:
    Gardiner_sequence,;occurrence_count;
    """
    unigrams = defaultdict(int)
    bigrams = defaultdict(int)
    trigrams = defaultdict(int)
    
    with open(ngrams_path, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # Split by semicolon pairs
    entries = content.split(';')
    
    i = 0
    while i < len(entries) - 1:
        sequence = entries[i].strip()
        count_str = entries[i + 1].strip()
        
        if sequence and count_str:
            try:
                count = int(count_str)
                gardiner_codes = [g.strip() for g in sequence.split(',') if g.strip()]
                
                if len(gardiner_codes) == 1:
                    unigrams[gardiner_codes[0]] += count
                elif len(gardiner_codes) == 2:
                    bigrams[tuple(gardiner_codes)] += count
                elif len(gardiner_codes) == 3:
                    trigrams[tuple(gardiner_codes)] += count
            except ValueError:
                pass
        
        i += 2
    
    return {
        'unigrams': dict(unigrams),
        'bigrams': dict(bigrams),
        'trigrams': dict(trigrams)
    }

# Load n-grams (update path as needed)
ngrams_path = "../../heiro_v11/data/raw/nGrams.txt"  # Adjust path
try:
    ngrams = parse_ngrams(ngrams_path)
    
    print(f"N-grams loaded:")
    print(f"  - Unigrams: {len(ngrams['unigrams']):,}")
    print(f"  - Bigrams: {len(ngrams['bigrams']):,}")
    print(f"  - Trigrams: {len(ngrams['trigrams']):,}")
    
    # Show most frequent unigrams
    print("\nMost frequent Gardiner signs:")
    top_unigrams = sorted(ngrams['unigrams'].items(), key=lambda x: x[1], reverse=True)[:10]
    for sign, count in top_unigrams:
        print(f"  {sign}: {count:,}")
    
    # Show most frequent bigrams
    print("\nMost frequent bigrams:")
    top_bigrams = sorted(ngrams['bigrams'].items(), key=lambda x: x[1], reverse=True)[:10]
    for bigram, count in top_bigrams:
        print(f"  {' → '.join(bigram)}: {count:,}")
        
except FileNotFoundError:
    print(f"⚠ N-grams file not found at {ngrams_path}")
    print("Please download from HamdiJr/Egyptian_hieroglyphs dataset")
    ngrams = None

N-grams loaded:
  - Unigrams: 805
  - Bigrams: 14,692
  - Trigrams: 51,727

Most frequent Gardiner signs:
  N35: 13,502
  X1: 12,759
  M17: 11,813
  Z1: 8,517
  D21: 8,358
  Z7: 7,317
  G1: 6,037
  G17: 5,240
  I9: 4,703
  A1: 4,056

Most frequent bigrams:
  M17 → M17: 2,429
  G41 → G1: 1,891
  M17 → Z7: 1,738
  D2 → Z1: 1,692
  Z1 → Z1: 1,488
  X1 → Z7: 1,402
  G1 → M17: 1,355
  N35 → X1: 1,333
  I10 → D46: 1,130
  X1 → Z1: 875


# Cell 6: Clean and Preprocess BBAW Data

In [7]:


def clean_transcription(text):
    """
    Remove philological markers:
    () defective, [] lost, {} surplus, 〈〉 omitted, 
    ⸢⸣ damaged, ⸮? unclear, {{}} erasure, etc.
    """
    if pd.isna(text):
        return ""
    
    # Remove all bracket types and their contents or just brackets
    text = re.sub(r'\[\[.*?\]\]', '', text)  # [[overstrike]]
    text = re.sub(r'\{\{.*?\}\}', '', text)  # {{erasure}}
    text = re.sub(r'\(\(.*?\)\)', '', text)  # ((above))
    text = re.sub(r'〈〈.*?〉〉', '', text)     # 〈〈haplography〉〉
    
    # Remove brackets but keep content
    text = re.sub(r'[\[\](){}〈〉⸢⸣⸮]', '', text)
    
    # Clean up extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def clean_translation(text):
    """Clean German translation text"""
    if pd.isna(text):
        return ""
    
    # Remove philological markers
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'\{.*?\}', '', text)
    
    # Clean up
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning to BBAW
print("Cleaning BBAW dataset...")
bbaw_df['transcription_clean'] = bbaw_df['transcription'].apply(clean_transcription)
bbaw_df['translation_clean'] = bbaw_df['translation'].apply(clean_translation)

# Filter out empty entries
bbaw_clean = bbaw_df[
    (bbaw_df['transcription_clean'].str.len() > 0) & 
    (bbaw_df['translation_clean'].str.len() > 0)
].copy()

print(f"BBAW after cleaning: {len(bbaw_clean):,} entries")

# Show before/after
print("\nCleaning example:")
idx = 100
print(f"Original transcription: {bbaw_df.iloc[idx]['transcription']}")
print(f"Cleaned transcription:  {bbaw_df.iloc[idx]['transcription_clean']}")
print(f"Original translation: {bbaw_df.iloc[idx]['translation']}")
print(f"Cleaned translation:  {bbaw_df.iloc[idx]['translation_clean']}")

Cleaning BBAW dataset...
BBAW after cleaning: 99,821 entries

Cleaning example:
Original transcription: sfḫ qn =k   
Cleaned transcription:  sfḫ qn =k
Original translation: Lass (wörtl.: wehre ab) deine Böswilligkeit (hinter dir)! ...
Cleaned translation:  Lass deine Böswilligkeit ! ...


# Cell 7: Clean and Preprocess TLA Data

In [8]:


# TLA may have (( )) in transliteration - optionally remove or keep
print("Cleaning TLA dataset...")
tla_df['transliteration_clean'] = tla_df['transliteration'].apply(
    lambda x: re.sub(r'\(\((.*?)\)\)', r'\1', str(x))  # Keep content, remove (( ))
)
tla_df['translation_clean'] = tla_df['translation'].apply(clean_translation)

# Filter out empty entries
tla_clean = tla_df[
    (tla_df['transliteration_clean'].str.len() > 0) & 
    (tla_df['translation_clean'].str.len() > 0)
].copy()

print(f"TLA after cleaning: {len(tla_clean):,} entries")

# Show example
print("\nTLA example:")
sample = tla_clean.iloc[50]
print(f"Transliteration: {sample['transliteration_clean']}")
print(f"Translation (DE): {sample['translation_clean']}")
print(f"Lemmas: {sample['lemmatization']}")

Cleaning TLA dataset...
TLA after cleaning: 12,768 entries

TLA example:
Transliteration: ḥmsi̯
Translation (DE): Setzt euch!
Lemmas: 105780|ḥmsi̯


# Cell 6b: Process Ramses Data - Egyptian Corpus Only


i addded this later and we need the prepare_corpus fucntion form cell 9 to run it... fix this later


In [14]:


if ramses_df is not None:
    print("\nProcessing Ramses Online data...")
    
    # Clean transliterations (similar to BBAW)
    ramses_df['transliteration_clean'] = ramses_df['transliteration'].apply(clean_transcription)
    
    # Filter out empty/very short entries
    ramses_clean = ramses_df[
        ramses_df['transliteration_clean'].str.len() > 5
    ].copy()
    
    print(f"Ramses after cleaning: {len(ramses_clean):,} entries")
    
    # ⚠️ IMPORTANT DECISION: How to use French translations?
    
    # OPTION 1: Egyptian corpus augmentation only (RECOMMENDED for German-first approach)
    print("\n✓ Using Ramses for EGYPTIAN corpus augmentation only")
    print("  (French translations will be used later for multilingual analysis)")
    
    # Add to Egyptian corpus
    ramses_egyptian_corpus = prepare_corpus(
        ramses_clean, 
        'transliteration_clean', 
        tokenize_egyptian
    )
    
    print(f"  Added {len(ramses_egyptian_corpus):,} Egyptian sentences from Ramses")
    
    # OPTION 2: French→German translation (NOT RECOMMENDED - adds noise)
    # You could use a translation API here, but it's better to keep French separate
    
else:
    ramses_clean = None
    ramses_egyptian_corpus = []


Processing Ramses Online data...
Ramses after cleaning: 9,063 entries

✓ Using Ramses for EGYPTIAN corpus augmentation only
  (French translations will be used later for multilingual analysis)


Preparing corpus:   0%|          | 0/9063 [00:00<?, ?it/s]

  Added 9,063 Egyptian sentences from Ramses


# Cell 8: Build Unified Vocabulary

In [15]:


def tokenize_egyptian(text):
    """Tokenize Egyptian transliteration"""
    # Split on spaces and common delimiters
    tokens = text.split()
    return [t for t in tokens if t and len(t) > 0]

def tokenize_german(text):
    """Simple German tokenization"""
    # Basic tokenization (consider using spaCy for production)
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text)
    return tokens

# Build Egyptian vocabulary from BBAW
print("Building Egyptian vocabulary...")
egyptian_vocab = Counter()

for text in tqdm(bbaw_clean['transcription_clean'], desc="BBAW"):
    tokens = tokenize_egyptian(text)
    egyptian_vocab.update(tokens)

# Add TLA vocabulary
for text in tqdm(tla_clean['transliteration_clean'], desc="TLA"):
    tokens = tokenize_egyptian(text)
    egyptian_vocab.update(tokens)

print(f"\nEgyptian vocabulary: {len(egyptian_vocab):,} unique tokens")
print(f"Total token count: {sum(egyptian_vocab.values()):,}")

# Build German vocabulary
print("\nBuilding German vocabulary...")
german_vocab = Counter()

for text in tqdm(bbaw_clean['translation_clean'], desc="BBAW DE"):
    tokens = tokenize_german(text)
    german_vocab.update(tokens)

for text in tqdm(tla_clean['translation_clean'], desc="TLA DE"):
    tokens = tokenize_german(text)
    german_vocab.update(tokens)

print(f"\nGerman vocabulary: {len(german_vocab):,} unique tokens")
print(f"Total token count: {sum(german_vocab.values()):,}")

# Show most common tokens
print("\nMost common Egyptian tokens:")
for token, count in egyptian_vocab.most_common(20):
    print(f"  {token}: {count:,}")

print("\nMost common German tokens:")
for token, count in german_vocab.most_common(20):
    print(f"  {token}: {count:,}")

Building Egyptian vocabulary...


BBAW:   0%|          | 0/99821 [00:00<?, ?it/s]

TLA:   0%|          | 0/12768 [00:00<?, ?it/s]


Egyptian vocabulary: 55,072 unique tokens
Total token count: 855,025

Building German vocabulary...


BBAW DE:   0%|          | 0/99821 [00:00<?, ?it/s]

TLA DE:   0%|          | 0/12768 [00:00<?, ?it/s]


German vocabulary: 36,263 unique tokens
Total token count: 1,202,604

Most common Egyptian tokens:
  =f: 42,727
  n: 37,371
  m: 34,587
  =k: 31,938
  =j: 18,670
  ḥr: 14,563
  r: 14,438
  jw: 9,243
  1: 7,403
  =s: 6,970
  =sn: 6,674
  nb: 5,590
  pꜣ: 5,272
  pw: 4,671
  2: 4,491
  n,j: 4,325
  sw: 4,128
  jr: 4,124
  jm: 4,069
  tꜣ: 4,057

Most common German tokens:
  der: 58,125
  des: 26,617
  die: 25,479
  ist: 21,716
  und: 20,711
  das: 15,247
  in: 15,245
  er: 13,634
  den: 13,133
  von: 12,395
  du: 12,343
  ich: 11,420
  zu: 11,401
  ein: 10,383
  dem: 8,831
  für: 8,582
  an: 8,578
  1: 8,104
  mit: 7,983
  sie: 7,804


# Cell 9: Create Training Corpus for Embeddings

In [16]:


def prepare_corpus(df, text_column, tokenizer):
    """Prepare tokenized corpus for embedding training"""
    corpus = []
    for text in tqdm(df[text_column], desc="Preparing corpus"):
        tokens = tokenizer(text)
        if len(tokens) > 0:
            corpus.append(tokens)
    return corpus

# Egyptian corpus (combined BBAW + TLA)
print("Preparing Egyptian training corpus...")
egyptian_corpus_bbaw = prepare_corpus(bbaw_clean, 'transcription_clean', tokenize_egyptian)
egyptian_corpus_tla = prepare_corpus(tla_clean, 'transliteration_clean', tokenize_egyptian)
egyptian_corpus = egyptian_corpus_bbaw + egyptian_corpus_tla

print(f"Egyptian corpus: {len(egyptian_corpus):,} sentences")
print(f"Average sentence length: {np.mean([len(s) for s in egyptian_corpus]):.1f} tokens")

# German corpus
print("\nPreparing German training corpus...")
german_corpus_bbaw = prepare_corpus(bbaw_clean, 'translation_clean', tokenize_german)
german_corpus_tla = prepare_corpus(tla_clean, 'translation_clean', tokenize_german)
german_corpus = german_corpus_bbaw + german_corpus_tla

print(f"German corpus: {len(german_corpus):,} sentences")
print(f"Average sentence length: {np.mean([len(s) for s in german_corpus]):.1f} tokens")


# Cell 9b: Update Egyptian Corpus with Ramses Data

# Egyptian corpus (combined BBAW + TLA + Ramses)
print("Preparing Egyptian training corpus...")
egyptian_corpus_bbaw = prepare_corpus(bbaw_clean, 'transcription_clean', tokenize_egyptian)
egyptian_corpus_tla = prepare_corpus(tla_clean, 'transliteration_clean', tokenize_egyptian)

# Add Ramses data (Late Egyptian)
if ramses_df is not None and ramses_egyptian_corpus:
    egyptian_corpus = egyptian_corpus_bbaw + egyptian_corpus_tla + ramses_egyptian_corpus
    print(f"  ✓ Including Ramses Late Egyptian: {len(ramses_egyptian_corpus):,} sentences")
else:
    egyptian_corpus = egyptian_corpus_bbaw + egyptian_corpus_tla

print(f"\nTotal Egyptian corpus: {len(egyptian_corpus):,} sentences")
print(f"  - BBAW (Middle Egyptian): {len(egyptian_corpus_bbaw):,}")
print(f"  - TLA (Earlier Egyptian): {len(egyptian_corpus_tla):,}")
if ramses_df is not None:
    print(f"  - Ramses (Late Egyptian): {len(ramses_egyptian_corpus):,}")

print(f"Average sentence length: {np.mean([len(s) for s in egyptian_corpus]):.1f} tokens")

# Note: German corpus stays the same (BBAW + TLA only)
print("\nGerman corpus: {len(german_corpus):,} sentences (BBAW + TLA only)")

Preparing Egyptian training corpus...


Preparing corpus:   0%|          | 0/99821 [00:00<?, ?it/s]

Preparing corpus:   0%|          | 0/12768 [00:00<?, ?it/s]

Egyptian corpus: 112,589 sentences
Average sentence length: 7.6 tokens

Preparing German training corpus...


Preparing corpus:   0%|          | 0/99821 [00:00<?, ?it/s]

Preparing corpus:   0%|          | 0/12768 [00:00<?, ?it/s]

German corpus: 112,284 sentences
Average sentence length: 10.7 tokens
Preparing Egyptian training corpus...


Preparing corpus:   0%|          | 0/99821 [00:00<?, ?it/s]

Preparing corpus:   0%|          | 0/12768 [00:00<?, ?it/s]

  ✓ Including Ramses Late Egyptian: 9,063 sentences

Total Egyptian corpus: 121,652 sentences
  - BBAW (Middle Egyptian): 99,821
  - TLA (Earlier Egyptian): 12,768
  - Ramses (Late Egyptian): 9,063
Average sentence length: 7.6 tokens

German corpus: {len(german_corpus):,} sentences (BBAW + TLA only)


# Cell 10: Data Summary and Validation

In [17]:


# print("="*60)
# print("DATA LOADING COMPLETE - SUMMARY")
# print("="*60)

# print(f"\n📊 Dataset Sizes:")
# print(f"  BBAW (cleaned):  {len(bbaw_clean):,} sentence pairs")
# print(f"  TLA (cleaned):   {len(tla_clean):,} sentence pairs")
# if lexicon_df is not None:
#     print(f"  Lexicon entries: {len(lexicon_df):,}")
# if ngrams is not None:
#     print(f"  N-grams:         {len(ngrams['unigrams']):,} unigrams, {len(ngrams['bigrams']):,} bigrams")

# print(f"\n📝 Vocabularies:")
# print(f"  Egyptian tokens: {len(egyptian_vocab):,} unique")
# print(f"  German tokens:   {len(german_vocab):,} unique")

# print(f"\n📚 Training Corpora:")
# print(f"  Egyptian: {len(egyptian_corpus):,} sentences")
# print(f"  German:   {len(german_corpus):,} sentences")

# print(f"\n✓ Ready for embedding training!")

# Cell 10: Updated Data Summary and Validation

print("="*60)
print("DATA LOADING COMPLETE - SUMMARY")
print("="*60)

print(f"\n📊 Dataset Sizes:")
print(f"  BBAW (cleaned):   {len(bbaw_clean):,} sentence pairs (Middle Egyptian)")
print(f"  TLA (cleaned):    {len(tla_clean):,} sentence pairs (Earlier Egyptian)")

if ramses_df is not None:
    print(f"  Ramses (cleaned): {len(ramses_clean):,} sentences (Late Egyptian) - Egyptian only")
if lexicon_df is not None:
    print(f"  Lexicon entries:  {len(lexicon_df):,}")
if ngrams is not None:
    print(f"  N-grams:          {len(ngrams['unigrams']):,} unigrams, {len(ngrams['bigrams']):,} bigrams")

print(f"\n📝 Vocabularies:")
print(f"  Egyptian tokens: {len(egyptian_vocab):,} unique")
print(f"  German tokens:   {len(german_vocab):,} unique")

print(f"\n📚 Training Corpora:")
print(f"  Egyptian: {len(egyptian_corpus):,} sentences")
if ramses_df is not None:
    print(f"    └─ Includes Late Egyptian from Ramses")
print(f"  German:   {len(german_corpus):,} sentences")
print(f"    └─ BBAW + TLA only (no French)")

print(f"\n💡 Strategy:")
print(f"  Primary alignment: Egyptian ↔ German")
print(f"  Ramses data: Egyptian corpus augmentation (increases vocabulary coverage)")
print(f"  French translations: Reserved for future multilingual experiments")

print(f"\n✓ Ready for embedding training!")

DATA LOADING COMPLETE - SUMMARY

📊 Dataset Sizes:
  BBAW (cleaned):   99,821 sentence pairs (Middle Egyptian)
  TLA (cleaned):    12,768 sentence pairs (Earlier Egyptian)
  Ramses (cleaned): 9,063 sentences (Late Egyptian) - Egyptian only
  Lexicon entries:  11,727
  N-grams:          805 unigrams, 14,692 bigrams

📝 Vocabularies:
  Egyptian tokens: 55,072 unique
  German tokens:   36,263 unique

📚 Training Corpora:
  Egyptian: 121,652 sentences
    └─ Includes Late Egyptian from Ramses
  German:   112,284 sentences
    └─ BBAW + TLA only (no French)

💡 Strategy:
  Primary alignment: Egyptian ↔ German
  Ramses data: Egyptian corpus augmentation (increases vocabulary coverage)
  French translations: Reserved for future multilingual experiments

✓ Ready for embedding training!


# Cell 11: Save Processed Data (Optional)

In [22]:

import pickle

# Create output directory
output_dir = Path("../data/processed/")
output_dir.mkdir(exist_ok=True)

# Save cleaned dataframes
bbaw_clean.to_parquet(output_dir / "bbaw_clean.parquet")
tla_clean.to_parquet(output_dir / "tla_clean.parquet")

if ramses_df is not None:
    ramses_clean.to_parquet(output_dir / "ramses_clean.parquet")

if lexicon_df is not None:
    lexicon_df.to_parquet(output_dir / "lexicon.parquet")

# Save vocabularies (updated with Ramses)
with open(output_dir / "egyptian_vocab.pkl", 'wb') as f:
    pickle.dump(egyptian_vocab, f)
    
with open(output_dir / "german_vocab.pkl", 'wb') as f:
    pickle.dump(german_vocab, f)

# Save corpora (Egyptian includes Ramses)
with open(output_dir / "egyptian_corpus.pkl", 'wb') as f:
    pickle.dump(egyptian_corpus, f)
    
with open(output_dir / "german_corpus.pkl", 'wb') as f:
    pickle.dump(german_corpus, f)

if ngrams is not None:
    with open(output_dir / "ngrams.pkl", 'wb') as f:
        pickle.dump(ngrams, f)

# Save metadata about corpus composition
corpus_metadata = {
    'bbaw_sentences': len(egyptian_corpus_bbaw),
    'tla_sentences': len(egyptian_corpus_tla),
    'ramses_sentences': len(ramses_egyptian_corpus) if ramses_df is not None else 0,
    'total_egyptian': len(egyptian_corpus),
    'total_german': len(german_corpus),
    'egyptian_vocab_size': len(egyptian_vocab),
    'german_vocab_size': len(german_vocab)
}

with open(output_dir / "corpus_metadata.json", 'w') as f:
    json.dump(corpus_metadata, f, indent=2)

print(f"✓ Processed data saved to {output_dir}/")
print(f"✓ Corpus metadata saved")

✓ Processed data saved to ../data/processed/
✓ Corpus metadata saved
